# Interactive Training Notebook for the Chess Engine

This notebook provides a simplified, single-threaded version of the training process for the chess engine. Unlike the `train.py` script which is optimized for performance with multiprocessing, this notebook is designed for experimentation, debugging, and understanding the core training loop in an interactive way.

**Note:** Running this notebook will be significantly slower than the main training script.

## 1. Setup and Imports

First, we import the necessary libraries and modules from our project.

In [ ]:
import os
import chess
import numpy as np
import tensorflow as tf
from stockfish import Stockfish

import config
from model import create_chess_model
from utils import board_to_tensor, move_to_index
from train import normalize_stockfish_eval # Re-use the normalization function

# Ensure the model directory exists
if not os.path.exists(config.MODEL_DIR):
    os.makedirs(config.MODEL_DIR)

## 2. Initialize Model, Optimizer, and Stockfish

Next, we create our chess model, set up the optimizer, and initialize the Stockfish engine. Make sure your `config.py` has the correct path to the Stockfish executable.

In [ ]:
# Load the model
chess_model = create_chess_model()
model_path = os.path.join(config.MODEL_DIR, config.MODEL_FILENAME)
if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}")
    chess_model.load_weights(model_path)
else:
    print("Initializing a new model.")

# Setup optimizer and loss functions
optimizer = tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE)
loss_fn_policy = tf.keras.losses.SparseCategoricalCrossentropy()
loss_fn_value = tf.keras.losses.MeanSquaredError()

# Initialize Stockfish
try:
    stockfish = Stockfish(path=config.STOCKFISH_PATH, parameters={"Skill Level": 5})
    print("Stockfish initialized successfully.")
except Exception as e:
    print(f"Error initializing Stockfish: {e}")
    print("Please ensure the path in config.py is correct.")

## 3. The Training Loop

This is the main part of the notebook. We'll loop for a specified number of games. In each game, the model will play against itself (or a simple opponent), and we'll use Stockfish to provide the "ground truth" for training.

We will collect data from each game and then perform a training step.

In [ ]:
NUM_GAMES_TO_PLAY = 5  # Set a small number for interactive training
game_data_buffer = []

print(f"Starting training for {NUM_GAMES_TO_PLAY} games...")

for game_num in range(NUM_GAMES_TO_PLAY):
    board = chess.Board()
    current_game_data = []
    print(f"\n--- Playing Game {game_num + 1}/{NUM_GAMES_TO_PLAY} ---")

    while not board.is_game_over(claim_draw=True):
        # Get model's move
        board_tensor = np.expand_dims(board_to_tensor(board), axis=0)
        policy, _ = chess_model.predict(board_tensor, verbose=0)
        
        legal_moves = list(board.legal_moves)
        legal_move_indices = [move_to_index(m) for m in legal_moves]
        legal_policy = np.zeros_like(policy[0])
        legal_policy[legal_move_indices] = policy[0][legal_move_indices]
        if np.sum(legal_policy) > 0:
            legal_policy /= np.sum(legal_policy)
            move = np.random.choice(legal_moves, p=legal_policy[legal_move_indices])
        else:
            move = np.random.choice(legal_moves)
        
        board.push(move)

        # Get Stockfish's evaluation and best move for the new position
        stockfish.set_fen_position(board.fen())
        sf_eval = stockfish.get_evaluation()
        sf_best_move_uci = stockfish.get_best_move()
        
        if sf_best_move_uci:
            sf_best_move = chess.Move.from_uci(sf_best_move_uci)
            sf_best_move_index = move_to_index(sf_best_move)
            normalized_eval = normalize_stockfish_eval(sf_eval)
            
            # Store the state *before* the move, with the targets from the *next* state
            current_game_data.append((board_tensor.squeeze(0), sf_best_move_index, normalized_eval))

    print(f"Game {game_num + 1} finished. Result: {board.result(claim_draw=True)}")
    game_data_buffer.extend(current_game_data)

    # Train on the collected data if we have enough
    if len(game_data_buffer) >= config.BATCH_SIZE:
        print(f"\n>>> Training on a batch of {config.BATCH_SIZE} samples...")
        
        indices = np.random.choice(len(game_data_buffer), config.BATCH_SIZE, replace=False)
        batch = [game_data_buffer[i] for i in indices]
        board_tensors, policy_targets, value_targets = zip(*batch)
        
        with tf.GradientTape() as tape:
            policy_preds, value_preds = chess_model(np.array(board_tensors))
            policy_loss = loss_fn_policy(policy_targets, policy_preds)
            value_loss = loss_fn_value(value_targets, value_preds)
            total_loss = policy_loss + value_loss
        
        grads = tape.gradient(total_loss, chess_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, chess_model.trainable_variables))
        
        print(f"Training step completed. Total Loss: {total_loss.numpy():.4f}")
        game_data_buffer.clear() # Clear buffer after training

print("\nInteractive training session finished.")

## 4. Save the Model

After the training session, we can save the updated model weights.

In [ ]:
chess_model.save_weights(model_path)
print(f"Model weights saved to {model_path}")